![DB Academy](./Includes/images/db-academy.png)

# Lakebase Autoscaling Branching

## Overview
This lecture provides an introduction to the Databricks Lakebase Autoscaling Branching feature, guiding you through the complete process of creating, configuring, and managing a Lakebase branche within the Databricks platform. 


## Learning Objectives
By the end of this lecture, you will understand:
- What Database Branching is
- Why Database Branches are needed during the software development lifecyle
- How to work with Branches in Databricks
- The different Branching Strategies

## A. BRIEF BACKGROUND

<img src="Includes/images/branching/branching-overview-intro-image-2.png"
     alt="Branching"
     width="1100">

Through the years, code has evolved to be agile (branches, PRs, CI/CD), but databases have stayed static. This means they don't match how teams like to build software. 

Teams need Databases to behave like code:

- Developers want isolated functional environments to test schema changes with no impact to production or other teams 
- CI/CD processes need fresh databases for every test run
- Preview environments should reflect real production data

Most databases today make all of this difficult to accomplish. The default solution to handle the above scenarios has always been, in one way or another, copying the database but this can get expensive, time-consuming, and error-prone.

As a result, teams compromise:

- They test schema changes against incomplete or stale data
- They share environments which can be dangerous

Lakebase, through **branching**, makes this process much easier!!

## B. BRANCHING OVERVIEW
<br>
<img src="Includes/images/branching/new-branching-overview-image.png"
     alt="Branching"
     width="1100">

### B1. What is Branching?

A branch in Lakebase Postgres is an independent database environment that is created from a parent branch or database instance. With Lakebase, you can instantly create these isolated environments for development, experimentation, or testing schema changes, without impacting production workloads or duplicating shared data unnecessarily.

<br>
<pre> 
production (root branch) \
    ├── staging (child of production)\
    │    └── feature-test (child of staging) \
    └── development (child of production) \
          └── bugfix-branch (child of development)
</pre>
<br>





#### B1.1 Copy on Write Storage

A branch shares storage with its parent branch through **copy-on-write**. When you create a branch, Lakebase takes the exact state (schema + data) of the parent branch at that moment, but it does not duplicate the entire database. 

The new branch will inherit both the schema and data from its parent, and they will share the same underlying storage. However, when the data is modified in the child branch, Lakebase will write new data. So the parent and child branches diverge independently to store only changes to the base schema made since branch creation.

**Benefits of copy-on-write storage:**
- Your branches appear instantly; the size of your database has no impact on branch creation time
- You only pay for data that actually changes between branches
- Creating branches has no performance impact on your production workload

<br>

<pre>
production branch         child branch (at creation)
┌─────────────────┐       ┌─────────────────┐
│  [Data A]       │◄──────│  → Data A       │  (shared)
│  [Data B]       │◄──────│  → Data B       │  (shared)
│  [Data C]       │◄──────│  → Data C       │  (shared)
└─────────────────┘       └─────────────────┘

After modifying data in child branch:
┌─────────────────┐       ┌─────────────────┐
│  [Data A]       │◄──────│  → Data A       │  (shared)
│  [Data B]       │       │  [Data B']      │  (changed)
│  [Data C]       │◄──────│  → Data C       │  (shared)
└─────────────────┘       └─────────────────┘
                          Only changed data is stored separately

</prev>

### B2. Working with Branches

By default, Lakebase creates a single production branch when you create a new project. **This is your default branch**, meant to house your application's production data. You can create additional branches as needed to fit your workflow. 

**By default, the default branch never scales to zero, however, this can be configured to scale to zero.**

#### B2.1 Creating Branches from Root / Production Branch

1. Navigate to your project's Branches page in the Lakebase App.

2. Locate and click the **New Branch 1/500** button in the upper right of the page

<img src="Includes/images/branching/create_branch_project_page.png"
     alt="create-branch-button"
     width="1000">

3. Enter a descriptive branch name (required)
4. Determine if you want to make your branch a non-expiring or an expiring branch _(**more details about this below in section 2.1.2**)_
4. Select Current data
5. Click Create to create the branch

<img src="Includes/images/branching/create_branch_current_data_expiration.png"
     alt="create-branch-from-production.png"
     width="700">

##### B2.1.1 Creating Branches from Past Data

You can create a branch from a specific point in time within your restore window. A restore window is a timeframe during which Lakebase retains a history of changes for all branches in your project. This is useful for recovering from data errors like accidental deletions, investigating past issues, or accessing historical data for auditing and compliance purposes. 

Example Scenarios
- A critical table was dropped yesterday at 10:23 AM, you can create a branch set to 10:22 AM to extract the missing data.
- Create branches reflecting your database state at specific dates for financial reconciliations, regulatory audits, or forensic analysis.

To learn more about point in time restore, refer to the following document [here](https://docs.databricks.com/aws/en/oltp/projects/point-in-time-restore)


To create a branch from past data:

1. Navigate to your project's Branches page in the Lakebase App.
2. Locate and click the New Branch 1/500 button in the upper right of the page

<img src="Includes/images/branching/create_branch_project_page.png"
     alt="create-branch-button"
     width="1000">

3. Enter a descriptive branch name (required)
4. Determine if you want to make your branch a non-expiring or an expiring branch **_(more details about this below in section 2.1.2)_**
4. Select Past Data
5. Use the date picker to select the desired point in time.
6. Click Create to create the branch

<img src="Includes/images/branching/create_branch_from_past_data.png"
     alt="create-branch-button"
     width="700">

##### B2.1.2 Expiring vs Non Expiring Branches

An exprining branch is one where you set an automatic deletion timestamps. When the branch reaches its expiration time, it is automatically deleted. This feature helps manage temporary branches and reduce storage costs.

Branch expiration is ideal for temporary branches that have predictable lifespans, such as:

- CI/CD environments: Test branches that should clean up after pipeline completion.
- Feature development: Time-boxed feature branches with known deadlines.
- Automated testing: Ephemeral test environments created by scripts.
- Development workflows: Temporary environments that don't need to persist indefinitely.


To learn more about how branch expiration works, please refer to the following document [here](https://docs.databricks.com/aws/en/oltp/projects/manage-branches#how-branch-expiration-works)

##### OTHER SPECIAL TYPES OF BRANCHES

<details>

**Protected branches**

These are branches with special rules that restrict certain operations like protection from deletion, being reset, archival and more!

Refer to [this document](https://docs.databricks.com/aws/en/oltp/projects/protected-branches) to learn more about protected branches

</details>

#### B2.2 Resetting a Branch to Match Parent State

When working with branches, you might find yourself in a situation where you need to update your working branch to the latest data from your parent branch.

When you reset a branch to its parent, the data and schema is completely replaced with the latest data and schema from its parent.

**When to use:**

- When a child branch is too far out of date with parent and you have no schema changes to consider or preserve; you just want a quick refresh of the data. You can perform a clean, instant reset to the latest data from the parent

##### KEY POINTS TO NOTE

<details>

- You can only reset a branch to the latest data from its parent. 
- Point-in-time resets based on timestamp are possible using point-in-time restore, a similar feature with some differences: _**point-in-time restore creates a new branch and is intended more for data recovery than development workflow.**_
- This reset is a complete overwrite, not a refresh or a merge. Any local changes made to the child branch are lost during this reset.
- Existing connections are temporarily interrupted during the reset. However, your connection details don't change. All connections are re-established as soon as the reset is done.
- Root branches (like your project's production branch) can't be reset because they have no parent branch to reset to.

</details>

To reset a branch to its parent:

- Navigate to your project's Branches page in the Lakebase App.
- Click the Kebab menu icon. menu next to the branch you want to reset and select Reset from parent.
- Confirm the reset operation.
<br>

![reset_branch.png](Includes/images/branching/reset_branch_to_parent_state.png)

## C. Schema Diff

### C1. What is the Schema Diff Tool?

Schema Diff tool lets you compare the schemas for two selected branches in a side-by-side view (or line-by-line on mobile devices). It shows a side-by-side SQL DDL comparison that highlights added, removed, or modified database objects such as tables, columns, indexes, and constraints.


It is meant to be used for pre-merge validation, development tracking, drift detection, and change documentation.

To compare schemas between branches:

1. Navigate to a child branch overview page in the Lakebase App.
2. In the Parent branch section, click Schema diff.


![child-branch-overview.png](Includes/images/branching/child-branch-overview-schema-diff-button.png)


3. Once the Schema Diff dialog pops up, select the base branch for comparison (defaults to parent branch).
4. On the Schema Diff dialog, select the database to compare.
5. Select the branch to compare against base (defaults to current child branch).
6. Click Compare.


![schema-diff-results.png](Includes/images/branching/schema-diff-lecture.png)

#### C2.1 Understanding the Schema Diff Results

- Red lines show what was removed or changed from the base branch
Green lines show what was added or changed in the compare branch.
- The diff captures changes to table definitions, columns, constraints, indexes, and other database objects.

If no differences exist between the selected branches, you see a success message confirming the schemas are in sync:

![no-schema-diff-results.png](Includes/images/branching/schema-diff-no-diffs.png)



### C2. Practical Applications

- Pre-Migration Check: Before migrating schemas from a development branch into main, use Schema Diff to ensure only intended schema changes are applied.
- Audit Changes: Historically compare schema changes to understand the evolution of your database structure.
- Consistency Checks: Ensure environment consistency by comparing schemas across development, staging, and production branches.


## D. Branch Strategies

Lets take a look at some common ways teams organize their branches:

### D1. Production - Development - Staging 

In this workflow, your development branch is where you build new features safely. You can make schema changes, add test data, and experiment without any risk to your production branch. When you're ready, run your tested schema migrations against production (using your migration tool), then reset development to start the next feature with fresh data.

If you need pre-production testing, maintain a staging branch that mirrors your production branch data. Deploy your application there, run integration and performance tests against realistic data, and gain confidence before going live. Periodically reset staging from production to refresh your test data.

<br>

<pre>
production
├── staging
└── development
</pre>

### D2. Per Developer Set Up

This pattern prevents developers from interfering with each other's work and allows everyone to test schema changes independently. Each developer can experiment with their own schema changes and data modifications without affecting others, then apply tested migrations to the shared development or production branch when ready.

<br>
<pre>
production
└── development
    ├── dev-alice
    ├── dev-bob
    └── dev-charlie
</pre>

### D3. And more.... Demo Time!!!

- Lets take a look at how to implement these branching strategies!